In [5]:

import os

dataset_path = r"C:\Users\HP\OneDrive\Desktop\HeavySuppliersWarehouseDatasets"

print("My current path:", dataset_path)
print("Folder exists:", os.path.isdir(dataset_path))




My current path: C:\Users\HP\OneDrive\Desktop\HeavySuppliersWarehouseDatasets
Folder exists: True


In [11]:

dataset_path = os.path.join(
    dataset_path,
    "HeavySuppliersWarehouseDatasets"
)

print("Folder exists:", os.path.isdir(dataset_path))
print("Files found:", len(os.listdir(dataset_path)))

for file in os.listdir(dataset_path):
    print(file)



Folder exists: True
Files found: 12
branches.csv
customers.csv
inventory_master.csv
invoices.csv
payments.csv
products.csv
purchase_orders_header.csv
purchase_orders_lines.csv
sales_orders_header.csv
sales_orders_lines.csv
stock_ledger.csv
suppliers.csv


In [8]:

import os

print("Files and folders inside dataset folder:\n")

for item in os.listdir(dataset_path):
    print(item)


Files and folders inside dataset folder:

HeavySuppliersWarehouseDatasets


In [12]:

# Load all 12 CSV datasets

datasets = {}

for file in os.listdir(dataset_path):
    if file.lower().endswith(".csv"):
        name = file[:-4]
        file_path = os.path.join(dataset_path, file)

        datasets[name] = pd.read_csv(file_path)

print("All datasets loaded successfully!")
print("Total datasets:", len(datasets))

# Display dataset dimensions
print("\nDATASET SUMMARY")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows and {df.shape[1]} columns")


All datasets loaded successfully!
Total datasets: 12

DATASET SUMMARY
branches: 6 rows and 13 columns
customers: 500 rows and 15 columns
inventory_master: 180 rows and 8 columns
invoices: 18033 rows and 10 columns
payments: 19257 rows and 5 columns
products: 30 rows and 23 columns
purchase_orders_header: 24000 rows and 10 columns
purchase_orders_lines: 155495 rows and 9 columns
sales_orders_header: 20000 rows and 11 columns
sales_orders_lines: 130402 rows and 9 columns
stock_ledger: 237230 rows and 9 columns
suppliers: 8 rows and 12 columns


In [13]:

# Display all column names from every dataset

for name, df in datasets.items():

    print("\n" + "=" * 60)
    print("DATASET:", name)
    print("=" * 60)

    print("Total Columns:", len(df.columns))

    for column in df.columns:
        print("-", column)



DATASET: branches
Total Columns: 13
- branch_id
- branch_name
- city
- state
- region
- warehouse_type
- warehouse_capacity
- service_center_available
- manager_id
- total_employees
- avg_monthly_revenue
- monthly_operational_cost
- market_demand_index

DATASET: customers
Total Columns: 15
- customer_id
- customer_type
- industry_segment
- city
- state
- pincode
- region
- branch_id
- credit_limit
- current_balance
- payment_terms
- customer_since
- last_purchase_date
- total_purchase_value
- customer_rating

DATASET: inventory_master
Total Columns: 8
- product_id
- branch_id
- opening_stock
- reorder_level
- safety_stock
- max_stock
- current_stock
- warehouse_bin

DATASET: invoices
Total Columns: 10
- invoice_id
- so_id
- customer_id
- branch_id
- invoice_date
- due_date
- total_order_value
- total_gst_amount
- grand_total
- payment_status

DATASET: payments
Total Columns: 5
- payment_id
- invoice_id
- payment_date
- payment_amount
- payment_method

DATASET: products
Total Columns: 

In [14]:

# Display column names and their data types

for name, df in datasets.items():

    print("\nDATASET:", name)
    print("-" * 50)

    print(df.dtypes)



DATASET: branches
--------------------------------------------------
branch_id                   object
branch_name                 object
city                        object
state                       object
region                      object
warehouse_type              object
warehouse_capacity          object
service_center_available    object
manager_id                   int64
total_employees              int64
avg_monthly_revenue          int64
monthly_operational_cost     int64
market_demand_index          int64
dtype: object

DATASET: customers
--------------------------------------------------
customer_id             object
customer_type           object
industry_segment        object
city                    object
state                   object
pincode                  int64
region                  object
branch_id               object
credit_limit             int64
current_balance          int64
payment_terms           object
customer_since          object
last_purchase_date

In [15]:

# Candidate primary keys for our datasets

primary_keys = {
    "branches": ["branch_id"],
    "customers": ["customer_id"],
    "inventory_master": ["product_id", "branch_id"],
    "invoices": ["invoice_id"],
    "payments": ["payment_id"],
    "products": ["product_id"],
    "purchase_orders_header": ["po_id"],
    "purchase_orders_lines": ["po_id", "line_number"],
    "sales_orders_header": ["so_id"],
    "sales_orders_lines": ["so_id", "line_number"],
    "suppliers": ["supplier_id"],
    "stock_ledger": ["ledger_id"]
}

# Validate primary keys

for name, keys in primary_keys.items():

    if name not in datasets:
        print(name, "- Dataset not found")
        continue

    df = datasets[name]

    if not all(key in df.columns for key in keys):
        print(name, "- Check key columns:", keys)
        continue

    missing = df[keys].isnull().any(axis=1).sum()
    duplicates = df.duplicated(subset=keys).sum()

    print("\nDataset:", name)
    print("Candidate key:", keys)
    print("Missing key records:", missing)
    print("Duplicate key records:", duplicates)



Dataset: branches
Candidate key: ['branch_id']
Missing key records: 0
Duplicate key records: 0

Dataset: customers
Candidate key: ['customer_id']
Missing key records: 0
Duplicate key records: 0

Dataset: inventory_master
Candidate key: ['product_id', 'branch_id']
Missing key records: 0
Duplicate key records: 0

Dataset: invoices
Candidate key: ['invoice_id']
Missing key records: 0
Duplicate key records: 197

Dataset: payments
Candidate key: ['payment_id']
Missing key records: 0
Duplicate key records: 202

Dataset: products
Candidate key: ['product_id']
Missing key records: 0
Duplicate key records: 0

Dataset: purchase_orders_header
Candidate key: ['po_id']
Missing key records: 0
Duplicate key records: 0

Dataset: purchase_orders_lines
Candidate key: ['po_id', 'line_number']
Missing key records: 0
Duplicate key records: 0

Dataset: sales_orders_header
Candidate key: ['so_id']
Missing key records: 0
Duplicate key records: 0

Dataset: sales_orders_lines
Candidate key: ['so_id', 'line_num

In [16]:

# Candidate foreign key relationships

relationships = [
    ("customers", "customer_id", "sales_orders_header", "customer_id"),
    ("products", "product_id", "inventory_master", "product_id"),
    ("products", "product_id", "sales_orders_lines", "product_id"),
    ("suppliers", "supplier_id", "purchase_orders_header", "supplier_id"),
    ("branches", "branch_id", "inventory_master", "branch_id"),
    ("branches", "branch_id", "sales_orders_header", "branch_id"),
    ("purchase_orders_header", "po_id", "purchase_orders_lines", "po_id"),
    ("sales_orders_header", "so_id", "sales_orders_lines", "so_id"),
    ("invoices", "invoice_id", "payments", "invoice_id")
]

for parent, parent_key, child, child_key in relationships:

    parent_df = datasets[parent]
    child_df = datasets[child]

    parent_ids = parent_df[parent_key].dropna().unique()

    unmatched = (
        child_df[child_key].notna()
        & ~child_df[child_key].isin(parent_ids)
    ).sum()

    print(
        f"{parent} -> {child}: "
        f"{unmatched} unmatched records"
    )


customers -> sales_orders_header: 0 unmatched records
products -> inventory_master: 0 unmatched records
products -> sales_orders_lines: 0 unmatched records
suppliers -> purchase_orders_header: 0 unmatched records
branches -> inventory_master: 0 unmatched records
branches -> sales_orders_header: 0 unmatched records
purchase_orders_header -> purchase_orders_lines: 0 unmatched records
sales_orders_header -> sales_orders_lines: 0 unmatched records
invoices -> payments: 0 unmatched records


In [17]:

# Check missing values in all datasets

missing_summary = []

for name, df in datasets.items():

    total_missing = df.isnull().sum().sum()

    missing_summary.append({
        "Dataset": name,
        "Total_Rows": len(df),
        "Total_Columns": len(df.columns),
        "Missing_Values": total_missing
    })

missing_df = pd.DataFrame(missing_summary)

print(missing_df.to_string(index=False))


               Dataset  Total_Rows  Total_Columns  Missing_Values
              branches           6             13               0
             customers         500             15               0
      inventory_master         180              8               0
              invoices       18033             10               0
              payments       19257              5               0
              products          30             23               0
purchase_orders_header       24000             10            2370
 purchase_orders_lines      155495              9               0
   sales_orders_header       20000             11               0
    sales_orders_lines      130402              9               0
          stock_ledger      237230              9               0
             suppliers           8             12               0


In [18]:

for name, df in datasets.items():

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    if len(missing) > 0:
        print("\nDataset:", name)
        print(missing)



Dataset: purchase_orders_header
received_date    2370
dtype: int64


In [19]:

# Investigate missing received dates

po_df = datasets["purchase_orders_header"]

missing_received = po_df[
    po_df["received_date"].isnull()
]

print("Missing received dates:",
      len(missing_received))

print("\nPurchase order status:")
print(missing_received["po_status"].value_counts(dropna=False))

print("\nOverall purchase order status:")
print(po_df["po_status"].value_counts(dropna=False))


Missing received dates: 2370

Purchase order status:
po_status
Cancelled    2370
Name: count, dtype: int64

Overall purchase order status:
po_status
Received     21630
Cancelled     2370
Name: count, dtype: int64


In [20]:

# Check complete duplicate rows in all datasets

for name, df in datasets.items():

    duplicate_count = df.duplicated().sum()

    print(
        f"{name}: {duplicate_count} duplicate rows"
    )


branches: 0 duplicate rows
customers: 0 duplicate rows
inventory_master: 0 duplicate rows
invoices: 0 duplicate rows
payments: 0 duplicate rows
products: 0 duplicate rows
purchase_orders_header: 0 duplicate rows
purchase_orders_lines: 0 duplicate rows
sales_orders_header: 0 duplicate rows
sales_orders_lines: 0 duplicate rows
stock_ledger: 0 duplicate rows
suppliers: 0 duplicate rows


In [21]:

# Identify columns that may contain dates

for name, df in datasets.items():

    date_columns = [
        col for col in df.columns
        if "date" in col.lower()
    ]

    if date_columns:
        print("\nDataset:", name)

        for col in date_columns:
            print(
                col,
                "->",
                df[col].dtype
            )



Dataset: customers
last_purchase_date -> object

Dataset: invoices
invoice_date -> object
due_date -> object

Dataset: payments
payment_date -> object

Dataset: products
last_purchase_date -> object

Dataset: purchase_orders_header
order_date -> object
expected_delivery_date -> object
received_date -> object

Dataset: sales_orders_header
order_date -> object
delivery_date -> object

Dataset: stock_ledger
movement_date -> object


In [22]:

# Create cleaned copies of all datasets

cleaned_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

# Date columns to convert

date_columns = {
    "customers": ["last_purchase_date"],
    "invoices": ["invoice_date", "due_date"],
    "payments": ["payment_date"],
    "products": ["last_purchase_date"],
    "purchase_orders_header": [
        "order_date",
        "expected_delivery_date",
        "received_date"
    ],
    "sales_orders_header": [
        "order_date",
        "delivery_date"
    ],
    "stock_ledger": ["movement_date"]
}

# Convert date columns

for name, columns in date_columns.items():

    for col in columns:

        cleaned_datasets[name][col] = pd.to_datetime(
            cleaned_datasets[name][col],
            errors="coerce"
        )

print("Date conversion completed!")


Date conversion completed!


In [23]:

for name, columns in date_columns.items():

    print("\nDataset:", name)

    for col in columns:
        print(
            col,
            "->",
            cleaned_datasets[name][col].dtype
        )



Dataset: customers
last_purchase_date -> datetime64[ns]

Dataset: invoices
invoice_date -> datetime64[ns]
due_date -> datetime64[ns]

Dataset: payments
payment_date -> datetime64[ns]

Dataset: products
last_purchase_date -> datetime64[ns]

Dataset: purchase_orders_header
order_date -> datetime64[ns]
expected_delivery_date -> datetime64[ns]
received_date -> datetime64[ns]

Dataset: sales_orders_header
order_date -> datetime64[ns]
delivery_date -> datetime64[ns]

Dataset: stock_ledger
movement_date -> datetime64[ns]


In [24]:

# Check important numeric columns

numeric_checks = {
    "products": ["unit_cost", "unit_price"],
    "sales_orders_lines": ["quantity", "unit_price"],
    "purchase_orders_lines": ["quantity", "unit_cost"],
    "inventory_master": ["opening_stock", "current_stock",
                         "reorder_level", "safety_stock"],
    "branches": ["warehouse_capacity"]
}

for name, columns in numeric_checks.items():

    df = cleaned_datasets[name]

    print("\nDataset:", name)

    for col in columns:

        negative = (df[col] < 0).sum()
        zero = (df[col] == 0).sum()

        print(
            f"{col}: Negative = {negative}, Zero = {zero}"
        )



Dataset: products
unit_cost: Negative = 0, Zero = 0
unit_price: Negative = 0, Zero = 0

Dataset: sales_orders_lines
quantity: Negative = 0, Zero = 0
unit_price: Negative = 0, Zero = 0

Dataset: purchase_orders_lines
quantity: Negative = 0, Zero = 0
unit_cost: Negative = 0, Zero = 0

Dataset: inventory_master
opening_stock: Negative = 0, Zero = 0
current_stock: Negative = 0, Zero = 0
reorder_level: Negative = 0, Zero = 0
safety_stock: Negative = 0, Zero = 0

Dataset: branches


TypeError: '<' not supported between instances of 'str' and 'int'

In [25]:
TypeError: '<' not supported between instances of 'str' and 'int'

SyntaxError: invalid syntax (1928453869.py, line 1)

In [26]:

# Inspect warehouse capacity values

branches_df = cleaned_datasets["branches"]

print("Data type:")
print(branches_df["warehouse_capacity"].dtype)

print("\nWarehouse capacity values:")
print(branches_df["warehouse_capacity"].to_list())

print("\nUnique values:")
print(branches_df["warehouse_capacity"].unique())


Data type:
object

Warehouse capacity values:
['45230 sqft', '38750 sqft', '41890 sqft', '36210 sqft', '29880 sqft', '34120 sqft']

Unique values:
['45230 sqft' '38750 sqft' '41890 sqft' '36210 sqft' '29880 sqft'
 '34120 sqft']


In [27]:

print(cleaned_datasets["branches"].dtypes)


branch_id                   object
branch_name                 object
city                        object
state                       object
region                      object
warehouse_type              object
warehouse_capacity          object
service_center_available    object
manager_id                   int64
total_employees              int64
avg_monthly_revenue          int64
monthly_operational_cost     int64
market_demand_index          int64
dtype: object


In [28]:

branches_df = cleaned_datasets["branches"]

print("Warehouse capacity values:")
print(branches_df["warehouse_capacity"].to_list())

print("\nUnique values:")
print(branches_df["warehouse_capacity"].unique())


Warehouse capacity values:
['45230 sqft', '38750 sqft', '41890 sqft', '36210 sqft', '29880 sqft', '34120 sqft']

Unique values:
['45230 sqft' '38750 sqft' '41890 sqft' '36210 sqft' '29880 sqft'
 '34120 sqft']


In [29]:
# Create a new numeric column
cleaned_datasets["branches"]["warehouse_capacity_sqft"] = (
    cleaned_datasets["branches"]["warehouse_capacity"]
    .str.replace(" sqft", "", regex=False)
    .astype(float)
)

# Check the result
print(cleaned_datasets["branches"][
    ["warehouse_capacity", "warehouse_capacity_sqft"]
])

# Check data type
print(cleaned_datasets["branches"]["warehouse_capacity_sqft"].dtype)

  warehouse_capacity  warehouse_capacity_sqft
0         45230 sqft                  45230.0
1         38750 sqft                  38750.0
2         41890 sqft                  41890.0
3         36210 sqft                  36210.0
4         29880 sqft                  29880.0
5         34120 sqft                  34120.0
float64


In [30]:
capacity = cleaned_datasets["branches"]["warehouse_capacity_sqft"]

print("Negative values:", (capacity < 0).sum())
print("Zero values:", (capacity == 0).sum())
print("Missing values:", capacity.isna().sum())
print("Data type:", capacity.dtype)

Negative values: 0
Zero values: 0
Missing values: 0
Data type: float64


In [31]:
for name, df in cleaned_datasets.items():
    print(f"\n--- {name} ---")
    print(df.dtypes[df.dtypes == "object"])


--- branches ---
branch_id                   object
branch_name                 object
city                        object
state                       object
region                      object
warehouse_type              object
warehouse_capacity          object
service_center_available    object
dtype: object

--- customers ---
customer_id         object
customer_type       object
industry_segment    object
city                object
state               object
region              object
branch_id           object
payment_terms       object
customer_since      object
dtype: object

--- inventory_master ---
product_id       object
branch_id        object
warehouse_bin    object
dtype: object

--- invoices ---
invoice_id        object
so_id             object
customer_id       object
branch_id         object
payment_status    object
dtype: object

--- payments ---
payment_id        object
invoice_id        object
payment_method    object
dtype: object

--- products ---
product_id        

In [32]:
print("CUSTOMER SINCE")
print(cleaned_datasets["customers"]["customer_since"].head(10))
print("\nUnique customer_since formats:")
print(cleaned_datasets["customers"]["customer_since"].unique()[:10])

print("\nPRODUCT DIMENSIONS")
print(cleaned_datasets["products"]["dimensions_cm"].head(10))
print("\nUnique dimensions:")
print(cleaned_datasets["products"]["dimensions_cm"].unique()[:10])


CUSTOMER SINCE
0    2024-04-10
1    2022-01-19
2    2017-02-14
3    2017-12-20
4    2023-07-27
5    2021-10-14
6    2019-03-17
7    2015-06-24
8    2022-04-07
9    2015-12-30
Name: customer_since, dtype: object

Unique customer_since formats:
['2024-04-10' '2022-01-19' '2017-02-14' '2017-12-20' '2023-07-27'
 '2021-10-14' '2019-03-17' '2015-06-24' '2022-04-07' '2015-12-30']

PRODUCT DIMENSIONS
0     40x25x25
1       10x8x8
2     18x12x12
3    180x40x35
4     20x12x12
5     30x20x20
6       90x4x4
7       12x6x4
8     32x18x18
9     40x25x25
Name: dimensions_cm, dtype: object

Unique dimensions:
['40x25x25' '10x8x8' '18x12x12' '180x40x35' '20x12x12' '30x20x20' '90x4x4'
 '12x6x4' '32x18x18' '120x18x18']


In [33]:
cleaned_datasets["customers"]["customer_since"] = pd.to_datetime(
    cleaned_datasets["customers"]["customer_since"],
    errors="coerce"
)

print(cleaned_datasets["customers"]["customer_since"].dtype)
print("Missing dates:",
      cleaned_datasets["customers"]["customer_since"].isna().sum())

datetime64[ns]
Missing dates: 0


In [34]:
# Split dimensions into three separate columns
dimensions = cleaned_datasets["products"]["dimensions_cm"].str.split(
    "x", expand=True
)

# Create numeric columns
cleaned_datasets["products"]["length_cm"] = pd.to_numeric(
    dimensions[0], errors="coerce"
)

cleaned_datasets["products"]["width_cm"] = pd.to_numeric(
    dimensions[1], errors="coerce"
)

cleaned_datasets["products"]["height_cm"] = pd.to_numeric(
    dimensions[2], errors="coerce"
)

# Display results
print(cleaned_datasets["products"][
    ["dimensions_cm", "length_cm", "width_cm", "height_cm"]
].head(10))

  dimensions_cm  length_cm  width_cm  height_cm
0      40x25x25         40        25         25
1        10x8x8         10         8          8
2      18x12x12         18        12         12
3     180x40x35        180        40         35
4      20x12x12         20        12         12
5      30x20x20         30        20         20
6        90x4x4         90         4          4
7        12x6x4         12         6          4
8      32x18x18         32        18         18
9      40x25x25         40        25         25


In [35]:
cleaned_datasets["products"]["volume_cm3"] = (
    cleaned_datasets["products"]["length_cm"]
    * cleaned_datasets["products"]["width_cm"]
    * cleaned_datasets["products"]["height_cm"]
)

print(cleaned_datasets["products"][
    ["product_name", "length_cm", "width_cm",
     "height_cm", "volume_cm3"]
].head(10))

              product_name  length_cm  width_cm  height_cm  volume_cm3
0    Hydraulic Pump HP-300         40        25         25       25000
1  Engine Oil Filter OF-90         10         8          8         640
2        Air Filter AF-120         18        12         12        2592
3        Track Chain TC-45        180        40         35      252000
4     Fuel Injector FI-220         20        12         12        2880
5       Alternator ALT-500         30        20         20       12000
6     Hydraulic Hose HH-60         90         4          4        1440
7       Bucket Tooth BT-10         12         6          4         288
8     Starter Motor SM-700         32        18         18       10368
9      Turbocharger TB-900         40        25         25       25000


In [36]:
products_df = cleaned_datasets["products"]

dimension_columns = [
    "length_cm",
    "width_cm",
    "height_cm"
]

for col in dimension_columns:
    print(f"\n{col}")
    print("Missing:", products_df[col].isna().sum())
    print("Zero:", (products_df[col] == 0).sum())
    print("Negative:", (products_df[col] < 0).sum())

print("\nVolume validation")
print("Missing volume:", products_df["volume_cm3"].isna().sum())
print("Zero volume:", (products_df["volume_cm3"] == 0).sum())
print("Negative volume:", (products_df["volume_cm3"] < 0).sum())



length_cm
Missing: 0
Zero: 0
Negative: 0

width_cm
Missing: 0
Zero: 0
Negative: 0

height_cm
Missing: 0
Zero: 0
Negative: 0

Volume validation
Missing volume: 0
Zero volume: 0
Negative volume: 0


In [37]:
for name, df in cleaned_datasets.items():
    numeric_cols = df.select_dtypes(include="number").columns

    print(f"\n--- {name} ---")

    for col in numeric_cols:
        negative = (df[col] < 0).sum()

        if negative > 0:
            print(f"{col}: {negative} negative values")
            


--- branches ---

--- customers ---

--- inventory_master ---

--- invoices ---

--- payments ---

--- products ---

--- purchase_orders_header ---

--- purchase_orders_lines ---

--- sales_orders_header ---

--- sales_orders_lines ---

--- stock_ledger ---

--- suppliers ---


In [38]:
for name, df in cleaned_datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print(f"\n--- {name} ---")
        print(missing)
    else:
        print(f"\n--- {name} ---")
        print("No missing values")


--- branches ---
No missing values

--- customers ---
No missing values

--- inventory_master ---
No missing values

--- invoices ---
No missing values

--- payments ---
No missing values

--- products ---
No missing values

--- purchase_orders_header ---
received_date    2370
dtype: int64

--- purchase_orders_lines ---
No missing values

--- sales_orders_header ---
No missing values

--- sales_orders_lines ---
No missing values

--- stock_ledger ---
No missing values

--- suppliers ---
No missing values


In [39]:
po_df = cleaned_datasets["purchase_orders_header"]

print(pd.crosstab(
    po_df["po_status"],
    po_df["received_date"].isna(),
    margins=True
))

received_date  False  True    All
po_status                        
Cancelled          0  2370   2370
Received       21630     0  21630
All            21630  2370  24000


In [40]:
primary_keys = {
    "branches": "branch_id",
    "customers": "customer_id",
    "inventory_master": None,
    "invoices": "invoice_id",
    "payments": "payment_id",
    "products": "product_id",
    "purchase_orders_header": "po_id",
    "sales_orders_header": "so_id",
    "stock_ledger": "movement_id",
    "suppliers": "supplier_id"
}

for name, key in primary_keys.items():
    if key is not None:
        df = cleaned_datasets[name]

        print(
            f"{name} | Duplicate {key}:",
            df[key].duplicated().sum()
        )

branches | Duplicate branch_id: 0
customers | Duplicate customer_id: 0
invoices | Duplicate invoice_id: 197
payments | Duplicate payment_id: 202
products | Duplicate product_id: 0
purchase_orders_header | Duplicate po_id: 0
sales_orders_header | Duplicate so_id: 0
stock_ledger | Duplicate movement_id: 0
suppliers | Duplicate supplier_id: 0


In [41]:
# Find duplicate invoice records
invoices_df = cleaned_datasets["invoices"]

duplicate_invoices = invoices_df[
    invoices_df["invoice_id"].duplicated(keep=False)
].sort_values("invoice_id")

print("Total duplicate invoice rows:",
      len(duplicate_invoices))

print("\nSample duplicate invoice records:")
print(duplicate_invoices.head(20))

Total duplicate invoice rows: 393

Sample duplicate invoice records:
       invoice_id      so_id customer_id branch_id invoice_date   due_date  \
14917  INV-105711  SO-901766       C0208    KOL001   2021-11-20 2022-01-04   
8324   INV-105711  SO-549003       C0191    KOL001   2024-12-15 2025-01-14   
14643  INV-106596  SO-807417       C0281    KOL001   2024-01-25 2024-03-25   
9894   INV-106596  SO-399413       C0409    PUN001   2021-06-16 2021-07-31   
1712   INV-111054  SO-233671       C0072    PUN001   2023-04-11 2023-04-26   
1706   INV-111054  SO-316263       C0313    AHM001   2019-08-02 2019-10-01   
13853  INV-112235  SO-236576       C0220    KOL001   2021-04-17 2021-06-16   
946    INV-112235  SO-363636       C0198    AHM001   2021-01-08 2021-03-09   
13492  INV-112378  SO-911241       C0082    AHM001   2025-01-01 2025-03-02   
1609   INV-112378  SO-870768       C0104    KOL001   2019-04-15 2019-06-14   
14375  INV-124878  SO-170742       C0426    KOL001   2022-05-30 2022-07-1

In [42]:
payments_df = cleaned_datasets["payments"]

duplicate_payments = payments_df[
    payments_df["payment_id"].duplicated(keep=False)
].sort_values("payment_id")

print("Total duplicate payment rows:",
      len(duplicate_payments))

print("\nSample duplicate payment records:")
print(duplicate_payments.head(20))

Total duplicate payment rows: 403

Sample duplicate payment records:
       payment_id  invoice_id payment_date  payment_amount payment_method
19220  PAY-102690  INV-826669   2021-04-23      3837051.40    Credit Card
7121   PAY-102690  INV-196859   2021-12-29       520518.80           Cash
16981  PAY-105592  INV-963449   2023-02-23      2123180.75           Cash
11803  PAY-105592  INV-323176   2022-07-10      1019902.14         Cheque
15115  PAY-112476  INV-743633   2024-06-17      4544341.00    Credit Card
5761   PAY-112476  INV-374653   2024-05-16      1724913.40           Cash
1148   PAY-115474  INV-645256   2022-12-23       473838.80    Credit Card
3929   PAY-115474  INV-583030   2023-08-25      3764707.00    Credit Card
5117   PAY-115921  INV-301855   2022-01-06       353798.70            UPI
9009   PAY-115921  INV-121000   2021-05-08       246614.08           Cash
3460   PAY-117803  INV-450834   2024-05-31       519996.42    Credit Card
9851   PAY-117803  INV-613514   2019-08-22 

In [43]:
for name, key in [
    ("invoices", "invoice_id"),
    ("payments", "payment_id")
]:
    df = cleaned_datasets[name]

    duplicate_ids = df[
        df[key].duplicated(keep=False)
    ]

    print(f"\n--- {name.upper()} ---")
    print("Total records:", len(df))
    print("Unique IDs:", df[key].nunique())
    print("Repeated ID values:",
          df.loc[df[key].duplicated(keep=False), key].nunique())
    print("Extra duplicate occurrences:",
          df[key].duplicated().sum())
    print("Rows affected:", len(duplicate_ids))


--- INVOICES ---
Total records: 18033
Unique IDs: 17836
Repeated ID values: 196
Extra duplicate occurrences: 197
Rows affected: 393

--- PAYMENTS ---
Total records: 19257
Unique IDs: 19055
Repeated ID values: 201
Extra duplicate occurrences: 202
Rows affected: 403


In [3]:
cleaned_datasets = {
    name: df.copy() for name, df in datasets.items()
}

date_columns = {
    "customers": ["last_purchase_date"],
    "invoices": ["invoice_date", "due_date"],
    "payments": ["payment_date"],
    "products": ["last_purchase_date"],
    "purchase_orders_header": [
        "order_date", "expected_delivery_date", "received_date"
    ],
    "sales_orders_header": ["order_date", "delivery_date"],
    "stock_ledger": ["movement_date"]
}

for name, columns in date_columns.items():
    for col in columns:
        cleaned_datasets[name][col] = pd.to_datetime(
            cleaned_datasets[name][col],
            errors="coerce"
        )

print("Cleaned datasets recreated successfully.")

Cleaned datasets recreated successfully.


In [2]:
import os
import pandas as pd
import numpy as np

dataset_path = r"C:\Users\HP\OneDrive\Desktop\HeavySuppliersWarehouseDatasets\HeavySuppliersWarehouseDatasets"

datasets = {}

for file in os.listdir(dataset_path):
    if file.endswith(".csv"):
        name = file.replace(".csv", "")
        datasets[name] = pd.read_csv(os.path.join(dataset_path, file))

print("Total datasets loaded:", len(datasets))

Total datasets loaded: 12


In [4]:
branches_df = cleaned_datasets["branches"]

print("Warehouse capacity values:")
print(branches_df["warehouse_capacity"].to_list())

print("\nUnique values:")
print(branches_df["warehouse_capacity"].unique())

Warehouse capacity values:
['45230 sqft', '38750 sqft', '41890 sqft', '36210 sqft', '29880 sqft', '34120 sqft']

Unique values:
['45230 sqft' '38750 sqft' '41890 sqft' '36210 sqft' '29880 sqft'
 '34120 sqft']


In [5]:
cleaned_datasets["branches"]["warehouse_capacity"] = (
    cleaned_datasets["branches"]["warehouse_capacity"]
    .str.replace("sqft", "", regex=False)
    .str.strip()
    .astype(float)
)

print(cleaned_datasets["branches"]["warehouse_capacity"])
print("\nData type:")
print(cleaned_datasets["branches"]["warehouse_capacity"].dtype)

0    45230.0
1    38750.0
2    41890.0
3    36210.0
4    29880.0
5    34120.0
Name: warehouse_capacity, dtype: float64

Data type:
float64


In [6]:
capacity = cleaned_datasets["branches"]["warehouse_capacity"]

print("Negative values:", (capacity < 0).sum())
print("Zero values:", (capacity == 0).sum())
print("Minimum capacity:", capacity.min())
print("Maximum capacity:", capacity.max())

Negative values: 0
Zero values: 0
Minimum capacity: 29880.0
Maximum capacity: 45230.0


In [7]:
for name, df in cleaned_datasets.items():
    print("\n" + "=" * 60)
    print("DATASET:", name.upper())
    print("=" * 60)

    print(df.dtypes)


DATASET: BRANCHES
branch_id                    object
branch_name                  object
city                         object
state                        object
region                       object
warehouse_type               object
warehouse_capacity          float64
service_center_available     object
manager_id                    int64
total_employees               int64
avg_monthly_revenue           int64
monthly_operational_cost      int64
market_demand_index           int64
dtype: object

DATASET: CUSTOMERS
customer_id                     object
customer_type                   object
industry_segment                object
city                            object
state                           object
pincode                          int64
region                          object
branch_id                       object
credit_limit                     int64
current_balance                  int64
payment_terms                   object
customer_since                  object
last_purcha

In [8]:
dtype_summary = []

for name, df in cleaned_datasets.items():
    for column in df.columns:
        dtype_summary.append({
            "Dataset": name,
            "Column": column,
            "Data_Type": str(df[column].dtype)
        })

dtype_summary_df = pd.DataFrame(dtype_summary)

print("Total columns checked:", len(dtype_summary_df))
display(dtype_summary_df)

Total columns checked: 134


,Dataset,Column,Data_Type
0,branches,branch_id,object
1,branches,branch_name,object
2,branches,city,object
3,branches,state,object
4,branches,region,object
...,...,...,...
129,suppliers,pincode,int64
130,suppliers,lead_time_days,int64
131,suppliers,reliability_score,int64
132,suppliers,import_duty_rate,int64


In [9]:
validation_results = []

for name, df in cleaned_datasets.items():

    missing_values = df.isnull().sum().sum()

    empty_strings = (
        df.select_dtypes(include="object")
        .apply(lambda col: col.astype(str).str.strip().eq("").sum())
        .sum()
    )

    validation_results.append({
        "Dataset": name,
        "Missing_Values": missing_values,
        "Empty_Strings": empty_strings
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print("Total missing values:",
      validation_df["Missing_Values"].sum())

print("Total empty strings:",
      validation_df["Empty_Strings"].sum())

,Dataset,Missing_Values,Empty_Strings
0,branches,0,0
1,customers,0,0
2,inventory_master,0,0
3,invoices,0,0
4,payments,0,0
5,products,0,0
6,purchase_orders_header,2370,0
7,purchase_orders_lines,0,0
8,sales_orders_header,0,0
9,sales_orders_lines,0,0


Total missing values: 2370
Total empty strings: 0


In [10]:
infinite_results = []

for name, df in cleaned_datasets.items():

    numeric_df = df.select_dtypes(include=np.number)

    infinite_count = np.isinf(numeric_df).sum().sum()

    infinite_results.append({
        "Dataset": name,
        "Infinite_Values": infinite_count
    })

infinite_df = pd.DataFrame(infinite_results)

display(infinite_df)

print("Total infinite values:",
      infinite_df["Infinite_Values"].sum())

,Dataset,Infinite_Values
0,branches,0
1,customers,0
2,inventory_master,0
3,invoices,0
4,payments,0
5,products,0
6,purchase_orders_header,0
7,purchase_orders_lines,0
8,sales_orders_header,0
9,sales_orders_lines,0


Total infinite values: 0


In [11]:
business_checks = {}

# 1. Product price validation
products = cleaned_datasets["products"]

business_checks["Negative Unit Cost"] = (
    products["unit_cost"] < 0
).sum()

business_checks["Negative Unit Price"] = (
    products["unit_price"] < 0
).sum()

# 2. Customer financial validation
customers = cleaned_datasets["customers"]

business_checks["Negative Credit Limit"] = (
    customers["credit_limit"] < 0
).sum()

business_checks["Negative Customer Balance"] = (
    customers["current_balance"] < 0
).sum()

# 3. Branch employee validation
branches = cleaned_datasets["branches"]

business_checks["Negative Employees"] = (
    branches["total_employees"] < 0
).sum()

# 4. GST validation
business_checks["Negative Product GST"] = (
    products["gst_rate"] < 0
).sum()

# 5. Sales delivery date validation
sales = cleaned_datasets["sales_orders_header"]

business_checks["Delivery Before Order"] = (
    sales["delivery_date"] < sales["order_date"]
).sum()

# Display results
business_check_df = pd.DataFrame(
    list(business_checks.items()),
    columns=["Business Rule", "Invalid Records"]
)

display(business_check_df)

,Business Rule,Invalid Records
0,Negative Unit Cost,0
1,Negative Unit Price,0
2,Negative Credit Limit,0
3,Negative Customer Balance,0
4,Negative Employees,0
5,Negative Product GST,0
6,Delivery Before Order,0


In [12]:
inventory = cleaned_datasets["inventory_master"]

inventory_checks = {
    "Negative Opening Stock": (inventory["opening_stock"] < 0).sum(),

    "Negative Current Stock": (inventory["current_stock"] < 0).sum(),

    "Negative Reorder Level": (inventory["reorder_level"] < 0).sum(),

    "Negative Safety Stock": (inventory["safety_stock"] < 0).sum(),

    "Safety Stock Above Maximum": (
        inventory["safety_stock"] > inventory["max_stock"]
    ).sum(),

    "Reorder Level Above Maximum": (
        inventory["reorder_level"] > inventory["max_stock"]
    ).sum(),

    "Current Stock Above Maximum": (
        inventory["current_stock"] > inventory["max_stock"]
    ).sum()
}

inventory_check_df = pd.DataFrame(
    list(inventory_checks.items()),
    columns=["Inventory Rule", "Invalid Records"]
)

display(inventory_check_df)

,Inventory Rule,Invalid Records
0,Negative Opening Stock,0
1,Negative Current Stock,0
2,Negative Reorder Level,0
3,Negative Safety Stock,0
4,Safety Stock Above Maximum,0
5,Reorder Level Above Maximum,0
6,Current Stock Above Maximum,180


In [13]:
inventory = cleaned_datasets["inventory_master"]

# Find records where current stock exceeds maximum stock
excess_inventory = inventory[
    inventory["current_stock"] > inventory["max_stock"]
].copy()

# Calculate excess quantity
excess_inventory["excess_stock"] = (
    excess_inventory["current_stock"] - excess_inventory["max_stock"]
)

print("Total affected records:", len(excess_inventory))

display(
    excess_inventory[
        [
            "product_id",
            "branch_id",
            "current_stock",
            "max_stock",
            "excess_stock"
        ]
    ]
)

Total affected records: 180


,product_id,branch_id,current_stock,max_stock,excess_stock
0,P001,DEL001,115928,340,115588
1,P001,PUN001,109389,349,109040
2,P001,CHN001,99373,311,99062
3,P001,HYD001,108772,417,108355
4,P001,KOL001,103808,475,103333
...,...,...,...,...,...
175,P030,PUN001,112388,283,112105
176,P030,CHN001,105447,246,105201
177,P030,HYD001,104611,268,104343
178,P030,KOL001,103465,365,103100


In [14]:
print("Total excess stock units:",
      excess_inventory["excess_stock"].sum())

print("Average excess stock:",
      round(excess_inventory["excess_stock"].mean(), 2))

print("Maximum excess stock in one record:",
      excess_inventory["excess_stock"].max())

Total excess stock units: 19245350
Average excess stock: 106918.61
Maximum excess stock in one record: 120630


In [15]:
# Display first 15 affected records
display(excess_inventory.head(15))

# Summary of inventory quantities
print("\nInventory stock summary:")
display(
    inventory[
        ["opening_stock", "current_stock",
         "reorder_level", "safety_stock", "max_stock"]
    ].describe()
)

# Check maximum stock in the product master
products = cleaned_datasets["products"]

print("\nProduct maximum stock summary:")
display(products[["product_id", "max_stock_level"]].describe())

,product_id,branch_id,opening_stock,reorder_level,safety_stock,max_stock,current_stock,warehouse_bin,excess_stock
0,P001,DEL001,196,59,46,340,115928,C15,115588
1,P001,PUN001,198,70,43,349,109389,D16,109040
2,P001,CHN001,187,62,44,311,99373,A11,99062
3,P001,HYD001,226,71,65,417,108772,A10,108355
4,P001,KOL001,263,94,58,475,103808,F03,103333
5,P001,AHM001,236,77,53,455,106714,C20,106259
6,P002,DEL001,166,57,35,261,108628,A06,108367
7,P002,PUN001,113,41,27,180,101623,E01,101443
8,P002,CHN001,267,102,76,465,105885,B04,105420
9,P002,HYD001,219,78,64,383,121013,A02,120630



Inventory stock summary:


,opening_stock,current_stock,reorder_level,safety_stock,max_stock
count,180.000000,180.000000,180.0000,180.000000,180.000000
mean,183.161111,107240.366667,64.5500,45.205556,321.755556
std,63.441009,5341.085645,24.0554,16.873402,116.554283
min,80.000000,88853.000000,24.0000,16.000000,131.000000
25%,129.000000,103487.500000,44.0000,31.000000,230.000000
50%,178.500000,107790.500000,62.0000,43.500000,310.500000
75%,234.500000,110779.500000,82.2500,58.000000,415.500000
max,300.000000,121013.000000,119.0000,85.000000,582.000000



Product maximum stock summary:


,max_stock_level
count,30.000000
mean,104.833333
std,123.991124
min,8.000000
25%,22.750000
50%,40.000000
75%,150.000000
max,500.000000


In [16]:
stock_comparison = inventory.merge(
    products[["product_id", "max_stock_level"]],
    on="product_id",
    how="left"
)

stock_comparison["max_stock_difference"] = (
    stock_comparison["max_stock"]
    - stock_comparison["max_stock_level"]
)

display(stock_comparison[
    [
        "product_id",
        "branch_id",
        "current_stock",
        "max_stock",
        "max_stock_level",
        "max_stock_difference"
    ]
])

,product_id,branch_id,current_stock,max_stock,max_stock_level,max_stock_difference
0,P001,DEL001,115928,340,40,300
1,P001,PUN001,109389,349,40,309
2,P001,CHN001,99373,311,40,271
3,P001,HYD001,108772,417,40,377
4,P001,KOL001,103808,475,40,435
...,...,...,...,...,...,...
175,P030,PUN001,112388,283,25,258
176,P030,CHN001,105447,246,25,221
177,P030,HYD001,104611,268,25,243
178,P030,KOL001,103465,365,25,340


In [17]:
print("Current stock statistics:")
print(inventory["current_stock"].describe())

print("\nMaximum stock statistics:")
print(inventory["max_stock"].describe())

print("\nOpening stock statistics:")
print(inventory["opening_stock"].describe())

Current stock statistics:
count       180.000000
mean     107240.366667
std        5341.085645
min       88853.000000
25%      103487.500000
50%      107790.500000
75%      110779.500000
max      121013.000000
Name: current_stock, dtype: float64

Maximum stock statistics:
count    180.000000
mean     321.755556
std      116.554283
min      131.000000
25%      230.000000
50%      310.500000
75%      415.500000
max      582.000000
Name: max_stock, dtype: float64

Opening stock statistics:
count    180.000000
mean     183.161111
std       63.441009
min       80.000000
25%      129.000000
50%      178.500000
75%      234.500000
max      300.000000
Name: opening_stock, dtype: float64


In [18]:
print(
    "Records with current stock above 10,000:",
    (inventory["current_stock"] > 10000).sum()
)

print(
    "Records with current stock above 100,000:",
    (inventory["current_stock"] > 100000).sum()
)

print(
    "Total inventory records:",
    len(inventory)
)

Records with current stock above 10,000: 180
Records with current stock above 100,000: 162
Total inventory records: 180


In [19]:
stock_ledger = cleaned_datasets["stock_ledger"]

print("Stock Ledger Columns:")
print(stock_ledger.columns.tolist())

print("\nFirst 10 Stock Ledger Records:")
display(stock_ledger.head(10))

Stock Ledger Columns:
['movement_id', 'product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'reference_type', 'reference_id', 'running_balance']

First 10 Stock Ledger Records:


,movement_id,product_id,branch_id,movement_type,movement_date,quantity,reference_type,reference_id,running_balance
0,MOV-OUT-184742,P001,AHM001,OUT,2019-01-03,8,SO,SO-157767,228
1,MOV-OUT-208422,P001,AHM001,OUT,2019-01-05,12,SO,SO-168735,216
2,MOV-IN-111128,P001,AHM001,IN,2019-01-23,174,PO,PO-181094,390
3,MOV-OUT-159140,P001,AHM001,OUT,2019-01-25,12,SO,SO-947377,378
4,MOV-OUT-173693,P001,AHM001,OUT,2019-01-25,11,SO,SO-879896,367
5,MOV-IN-79354,P001,AHM001,IN,2019-01-31,80,PO,PO-108730,447
6,MOV-OUT-201160,P001,AHM001,OUT,2019-02-07,7,SO,SO-631247,440
7,MOV-IN-54193,P001,AHM001,IN,2019-02-08,51,PO,PO-985728,491
8,MOV-IN-54189,P001,AHM001,IN,2019-02-08,242,PO,PO-985728,733
9,MOV-IN-70161,P001,AHM001,IN,2019-02-10,158,PO,PO-959228,891


In [20]:
display(
    inventory[
        ["opening_stock", "current_stock",
         "reorder_level", "safety_stock", "max_stock"]
    ].describe()
)

,opening_stock,current_stock,reorder_level,safety_stock,max_stock
count,180.000000,180.000000,180.0000,180.000000,180.000000
mean,183.161111,107240.366667,64.5500,45.205556,321.755556
std,63.441009,5341.085645,24.0554,16.873402,116.554283
min,80.000000,88853.000000,24.0000,16.000000,131.000000
25%,129.000000,103487.500000,44.0000,31.000000,230.000000
50%,178.500000,107790.500000,62.0000,43.500000,310.500000
75%,234.500000,110779.500000,82.2500,58.000000,415.500000
max,300.000000,121013.000000,119.0000,85.000000,582.000000


In [21]:
stock_ledger = cleaned_datasets["stock_ledger"]

print("Stock Ledger Columns:")
print(stock_ledger.columns.tolist())

display(stock_ledger.head(10))

Stock Ledger Columns:
['movement_id', 'product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'reference_type', 'reference_id', 'running_balance']


,movement_id,product_id,branch_id,movement_type,movement_date,quantity,reference_type,reference_id,running_balance
0,MOV-OUT-184742,P001,AHM001,OUT,2019-01-03,8,SO,SO-157767,228
1,MOV-OUT-208422,P001,AHM001,OUT,2019-01-05,12,SO,SO-168735,216
2,MOV-IN-111128,P001,AHM001,IN,2019-01-23,174,PO,PO-181094,390
3,MOV-OUT-159140,P001,AHM001,OUT,2019-01-25,12,SO,SO-947377,378
4,MOV-OUT-173693,P001,AHM001,OUT,2019-01-25,11,SO,SO-879896,367
5,MOV-IN-79354,P001,AHM001,IN,2019-01-31,80,PO,PO-108730,447
6,MOV-OUT-201160,P001,AHM001,OUT,2019-02-07,7,SO,SO-631247,440
7,MOV-IN-54193,P001,AHM001,IN,2019-02-08,51,PO,PO-985728,491
8,MOV-IN-54189,P001,AHM001,IN,2019-02-08,242,PO,PO-985728,733
9,MOV-IN-70161,P001,AHM001,IN,2019-02-10,158,PO,PO-959228,891


In [23]:
# Sort ledger by date
ledger_sorted = stock_ledger.sort_values(
    ["product_id", "branch_id", "movement_date"]
)

# Get latest movement for each product and branch
latest_ledger = (
    ledger_sorted
    .groupby(["product_id", "branch_id"])
    .tail(1)
    [["product_id", "branch_id", "movement_date", "running_balance"]]
)

# Compare with inventory master
ledger_comparison = inventory.merge(
    latest_ledger,
    on=["product_id", "branch_id"],
    how="left"
)

ledger_comparison["balance_difference"] = (
    ledger_comparison["current_stock"]
    - ledger_comparison["running_balance"]
)

display(ledger_comparison.head(20))

,product_id,branch_id,opening_stock,reorder_level,safety_stock,max_stock,current_stock,warehouse_bin,movement_date,running_balance,balance_difference
0,P001,DEL001,196,59,46,340,115928,C15,2025-01-19,115928,0
1,P001,PUN001,198,70,43,349,109389,D16,2025-01-18,109389,0
2,P001,CHN001,187,62,44,311,99373,A11,2025-01-16,99373,0
3,P001,HYD001,226,71,65,417,108772,A10,2025-01-27,108772,0
4,P001,KOL001,263,94,58,475,103808,F03,2025-01-18,103808,0
5,P001,AHM001,236,77,53,455,106714,C20,2025-01-27,106714,0
6,P002,DEL001,166,57,35,261,108628,A06,2025-01-17,108628,0
7,P002,PUN001,113,41,27,180,101623,E01,2025-01-18,101623,0
8,P002,CHN001,267,102,76,465,105885,B04,2025-01-20,106182,-297
9,P002,HYD001,219,78,64,383,121013,A02,2025-01-12,121013,0


In [24]:
print("Total inventory records:", len(ledger_comparison))

print(
    "Matching balances:",
    (ledger_comparison["balance_difference"] == 0).sum()
)

print(
    "Different balances:",
    (ledger_comparison["balance_difference"] != 0).sum()
)

print(
    "Missing ledger balances:",
    ledger_comparison["running_balance"].isna().sum()
)

print(
    "Average balance difference:",
    round(ledger_comparison["balance_difference"].mean(), 2)
)

Total inventory records: 180
Matching balances: 166
Different balances: 14
Missing ledger balances: 0
Average balance difference: -15.16


In [25]:
mismatched_records = ledger_comparison[
    ledger_comparison["balance_difference"] != 0
].copy()

print("Mismatched records:", len(mismatched_records))

display(
    mismatched_records[
        [
            "product_id",
            "branch_id",
            "current_stock",
            "running_balance",
            "movement_date",
            "balance_difference"
        ]
    ]
)

Mismatched records: 14


,product_id,branch_id,current_stock,running_balance,movement_date,balance_difference
8,P002,CHN001,105885,106182,2025-01-20,-297
19,P004,PUN001,104859,105024,2025-01-18,-165
47,P008,AHM001,109281,109317,2025-01-24,-36
59,P010,AHM001,115149,115549,2025-01-24,-400
120,P021,DEL001,115902,116107,2025-01-17,-205
135,P023,HYD001,105798,105981,2025-01-15,-183
147,P025,HYD001,113613,114128,2025-01-13,-515
151,P026,PUN001,105190,105335,2025-01-14,-145
153,P026,HYD001,108348,108546,2025-01-18,-198
162,P028,DEL001,106744,106967,2025-01-23,-223


In [26]:
print("Latest ledger movement date:",
      stock_ledger["movement_date"].max())

print("\nInventory mismatch summary:")
print(
    mismatched_records["balance_difference"].describe()
)

Latest ledger movement date: 2025-01-28 00:00:00

Inventory mismatch summary:
count     14.000000
mean    -194.857143
std      142.234291
min     -515.000000
25%     -235.000000
50%     -190.500000
75%      -96.250000
max       16.000000
Name: balance_difference, dtype: float64


In [22]:
print("Stock Ledger Data Types:")
print(stock_ledger.dtypes)

print("\nStock Ledger Summary:")
display(stock_ledger.describe(include="all"))


Stock Ledger Data Types:
movement_id                object
product_id                 object
branch_id                  object
movement_type              object
movement_date      datetime64[ns]
quantity                    int64
reference_type             object
reference_id               object
running_balance             int64
dtype: object

Stock Ledger Summary:


,movement_id,product_id,branch_id,movement_type,movement_date,quantity,reference_type,reference_id,running_balance
count,237230,237230,237230,237230,237230,237230.000000,237230,237230,237230.000000
unique,237230,30,6,3,NaN,NaN,3,43709,NaN
top,MOV-OUT-184742,P012,CHN001,IN,NaN,NaN,PO,PO-143567,NaN
freq,1,8082,44029,127611,NaN,NaN,127611,12,NaN
mean,NaN,NaN,NaN,NaN,2022-01-08 12:31:01.079964928,91.007111,NaN,NaN,53439.071268
min,NaN,NaN,NaN,NaN,2019-01-01 00:00:00,1.000000,NaN,NaN,14.000000
25%,NaN,NaN,NaN,NaN,2020-07-15 00:00:00,11.000000,NaN,NaN,26531.000000
50%,NaN,NaN,NaN,NaN,2022-01-03 00:00:00,40.000000,NaN,NaN,53240.500000
75%,NaN,NaN,NaN,NaN,2023-07-13 00:00:00,170.000000,NaN,NaN,80015.000000
max,NaN,NaN,NaN,NaN,2025-01-28 00:00:00,300.000000,NaN,NaN,121013.000000


In [27]:
display(
    mismatched_records[
        [
            "product_id",
            "branch_id",
            "current_stock",
            "running_balance",
            "movement_date",
            "balance_difference"
        ]
    ].sort_values("balance_difference")
)

,product_id,branch_id,current_stock,running_balance,movement_date,balance_difference
147,P025,HYD001,113613,114128,2025-01-13,-515
59,P010,AHM001,115149,115549,2025-01-24,-400
8,P002,CHN001,105885,106182,2025-01-20,-297
172,P029,KOL001,105010,105249,2025-01-09,-239
162,P028,DEL001,106744,106967,2025-01-23,-223
120,P021,DEL001,115902,116107,2025-01-17,-205
153,P026,HYD001,108348,108546,2025-01-18,-198
135,P023,HYD001,105798,105981,2025-01-15,-183
19,P004,PUN001,104859,105024,2025-01-18,-165
151,P026,PUN001,105190,105335,2025-01-14,-145


In [28]:
# Check whether mismatches are concentrated in particular branches
display(
    mismatched_records.groupby("branch_id").agg(
        Mismatched_Records=("product_id", "count"),
        Average_Difference=("balance_difference", "mean")
    ).reset_index()
)

,branch_id,Mismatched_Records,Average_Difference
0,AHM001,2,-218.000000
1,CHN001,1,-297.000000
2,DEL001,3,-169.333333
3,HYD001,3,-298.666667
4,KOL001,1,-239.000000
5,PUN001,4,-88.000000


In [29]:
step18_summary = {
    "Total Inventory Records": 180,
    "Records Above Maximum Stock": 180,
    "Matching Ledger Balances": 166,
    "Mismatched Ledger Balances": 14,
    "Missing Ledger Balances": 0,
    "Total Excess Stock Units": 19245350,
    "Latest Ledger Date": "2025-01-28"
}

for key, value in step18_summary.items():
    print(f"{key}: {value}")


Total Inventory Records: 180
Records Above Maximum Stock: 180
Matching Ledger Balances: 166
Mismatched Ledger Balances: 14
Missing Ledger Balances: 0
Total Excess Stock Units: 19245350
Latest Ledger Date: 2025-01-28


In [33]:
sales_lines = cleaned_datasets["sales_orders_lines"]

print("Sales Order Line Columns:")
print(sales_lines.columns.tolist())

print("\nFirst 5 records:")
display(sales_lines.head())

Sales Order Line Columns:
['so_id', 'line_number', 'product_id', 'quantity', 'unit_price', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total']

First 5 records:


,so_id,line_number,product_id,quantity,unit_price,gst_rate,line_total,gst_amount,line_grand_total
0,SO-990591,1,P018,19,11300,28,214700,60116.0,274816.0
1,SO-990591,2,P026,2,310,18,620,111.6,731.6
2,SO-990591,3,P026,3,310,18,930,167.4,1097.4
3,SO-990591,4,P013,9,1450,18,13050,2349.0,15399.0
4,SO-990591,5,P004,3,39800,28,119400,33432.0,152832.0


In [34]:
sales_lines = cleaned_datasets["sales_orders_lines"]

# Calculate expected line total
sales_lines["calculated_line_total"] = (
    sales_lines["quantity"] * sales_lines["unit_price"]
)

# Calculate expected GST amount
sales_lines["calculated_gst"] = (
    sales_lines["line_total"] * sales_lines["gst_rate"] / 100
)

# Calculate expected grand total
sales_lines["calculated_grand_total"] = (
    sales_lines["line_total"] + sales_lines["gst_amount"]
)

# Check differences
print(
    "Line total mismatches:",
    (
        (sales_lines["line_total"] -
         sales_lines["calculated_line_total"]).abs() > 0.01
    ).sum()
)

print(
    "GST mismatches:",
    (
        (sales_lines["gst_amount"] -
         sales_lines["calculated_gst"]).abs() > 0.01
    ).sum()
)

print(
    "Grand total mismatches:",
    (
        (sales_lines["line_grand_total"] -
         sales_lines["calculated_grand_total"]).abs() > 0.01
    ).sum()
)

Line total mismatches: 0
GST mismatches: 0
Grand total mismatches: 0


In [35]:
purchase_lines = cleaned_datasets["purchase_orders_lines"]

print("Purchase Order Columns:")
print(purchase_lines.columns.tolist())

display(purchase_lines.head())

Purchase Order Columns:
['po_id', 'line_number', 'product_id', 'quantity', 'unit_cost', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total']


,po_id,line_number,product_id,quantity,unit_cost,gst_rate,line_total,gst_amount,line_grand_total
0,PO-141186,1,P004,234,34709.54,28,8122032.36,2.274169e+06,1.039620e+07
1,PO-141186,2,P022,79,12317.84,28,973109.36,2.724706e+05,1.245580e+06
2,PO-141186,3,P030,46,13840.45,28,636660.70,1.782650e+05,8.149257e+05
3,PO-141186,4,P001,293,16988.41,28,4977604.13,1.393729e+06,6.371333e+06
4,PO-141186,5,P014,126,65443.06,28,8245825.56,2.308831e+06,1.055466e+07


In [36]:
purchase_lines = cleaned_datasets["purchase_orders_lines"]

# Calculate expected line total
purchase_lines["calculated_line_total"] = (
    purchase_lines["quantity"] * purchase_lines["unit_cost"]
)

# Calculate expected GST
purchase_lines["calculated_gst"] = (
    purchase_lines["line_total"] * purchase_lines["gst_rate"] / 100
)

# Calculate expected grand total
purchase_lines["calculated_grand_total"] = (
    purchase_lines["line_total"] + purchase_lines["gst_amount"]
)

# Check mismatches
print(
    "Line total mismatches:",
    ((purchase_lines["line_total"] -
      purchase_lines["calculated_line_total"]).abs() > 0.01).sum()
)

print(
    "GST mismatches:",
    ((purchase_lines["gst_amount"] -
      purchase_lines["calculated_gst"]).abs() > 0.01).sum()
)

print(
    "Grand total mismatches:",
    ((purchase_lines["line_grand_total"] -
      purchase_lines["calculated_grand_total"]).abs() > 0.01).sum()
)

Line total mismatches: 0
GST mismatches: 0
Grand total mismatches: 0


In [37]:
invoices = cleaned_datasets["invoices"]

print("Invoice Columns:")
print(invoices.columns.tolist())

print("\nFirst 5 Invoice Records:")
display(invoices.head())

Invoice Columns:
['invoice_id', 'so_id', 'customer_id', 'branch_id', 'invoice_date', 'due_date', 'total_order_value', 'total_gst_amount', 'grand_total', 'payment_status']

First 5 Invoice Records:


,invoice_id,so_id,customer_id,branch_id,invoice_date,due_date,total_order_value,total_gst_amount,grand_total,payment_status
0,INV-325361,SO-990591,C0070,AHM001,2022-05-28,2022-06-27,1345720,374439.6,1720159.6,Unpaid
1,INV-568476,SO-431510,C0031,KOL001,2022-05-08,2022-06-07,2194850,609423.0,2804273.0,Unpaid
2,INV-786913,SO-495859,C0013,AHM001,2023-10-02,2023-11-01,2620800,733824.0,3354624.0,Paid
3,INV-697615,SO-745068,C0382,HYD001,2021-10-09,2021-11-08,1094950,292961.0,1387911.0,Paid
4,INV-747829,SO-175713,C0472,CHN001,2024-11-04,2024-12-19,1250,225.0,1475.0,Paid


In [38]:
invoice_check = invoices.merge(
    cleaned_datasets["sales_orders_header"][
        ["so_id", "total_order_value", "total_gst_amount", "grand_total"]
    ],
    on="so_id",
    how="left",
    suffixes=("_invoice", "_sales")
)

print("Total invoice records:", len(invoice_check))

print(
    "Missing matching sales orders:",
    invoice_check["total_order_value_sales"].isna().sum()
)

print(
    "Order value mismatches:",
    (
        (invoice_check["total_order_value_invoice"] -
         invoice_check["total_order_value_sales"]).abs() > 0.01
    ).sum()
)

print(
    "GST mismatches:",
    (
        (invoice_check["total_gst_amount_invoice"] -
         invoice_check["total_gst_amount_sales"]).abs() > 0.01
    ).sum()
)

print(
    "Grand total mismatches:",
    (
        (invoice_check["grand_total_invoice"] -
         invoice_check["grand_total_sales"]).abs() > 0.01
    ).sum()
)

Total invoice records: 18033
Missing matching sales orders: 0
Order value mismatches: 0
GST mismatches: 0
Grand total mismatches: 0


In [39]:
payments = cleaned_datasets["payments"]
invoices = cleaned_datasets["invoices"]

# Check payment-to-invoice relationship
payment_check = payments.merge(
    invoices[["invoice_id", "grand_total"]],
    on="invoice_id",
    how="left"
)

print("Total payment records:", len(payment_check))

print(
    "Payments without matching invoices:",
    payment_check["grand_total"].isna().sum()
)

print(
    "Zero or negative payment amounts:",
    (payment_check["payment_amount"] <= 0).sum()
)

print(
    "Payments exceeding invoice grand total:",
    (
        payment_check["payment_amount"] >
        payment_check["grand_total"] + 0.01
    ).sum()
)

print("\nPayment Amount Summary:")
display(payment_check["payment_amount"].describe())

Total payment records: 19678
Payments without matching invoices: 0
Zero or negative payment amounts: 0
Payments exceeding invoice grand total: 164

Payment Amount Summary:


count    1.967800e+04
mean     1.210846e+06
std      1.233991e+06
min      1.913000e+01
25%      2.691802e+05
50%      8.217800e+05
75%      1.771463e+06
max      9.970301e+06
Name: payment_amount, dtype: float64

In [40]:
# Display payments exceeding invoice value
excess_payments = payment_check[
    payment_check["payment_amount"] >
    payment_check["grand_total"] + 0.01
]

print("Excess payment records:", len(excess_payments))

display(
    excess_payments[
        ["invoice_id", "payment_amount", "grand_total"]
    ].head(15)
)

Excess payment records: 164


,invoice_id,payment_amount,grand_total
441,INV-603462,931608.60,609280.0
504,INV-632936,3085330.60,2186416.0
788,INV-173978,2541624.00,843761.4
878,INV-589052,268293.07,81892.0
1026,INV-112235,3023775.00,1885322.8
1094,INV-567842,513231.20,328690.0
1384,INV-861743,1744540.55,751672.2
1386,INV-181646,2053785.17,1150634.8
1523,INV-550128,1514617.60,486016.0
1579,INV-670479,2866480.00,1998659.0


In [41]:
# Aggregate payments for each invoice
payment_summary = (
    payment_check.groupby("invoice_id", as_index=False)
    .agg(
        total_paid=("payment_amount", "sum"),
        invoice_amount=("grand_total", "first"),
        payment_count=("payment_amount", "count")
    )
)

payment_summary["excess_amount"] = (
    payment_summary["total_paid"] -
    payment_summary["invoice_amount"]
)

print(
    "Invoices where total payments exceed invoice amount:",
    (payment_summary["excess_amount"] > 0.01).sum()
)

print(
    "Invoices with multiple payments:",
    (payment_summary["payment_count"] > 1).sum()
)

Invoices where total payments exceed invoice amount: 233
Invoices with multiple payments: 2562


In [42]:
overpaid_invoices = payment_summary[
    payment_summary["excess_amount"] > 0.01
].copy()

print("Overpaid invoice records:", len(overpaid_invoices))

display(
    overpaid_invoices.sort_values(
        "excess_amount",
        ascending=False
    ).head(15)
)

Overpaid invoice records: 233


,invoice_id,total_paid,invoice_amount,payment_count,excess_amount
9131,INV-612771,16192709.20,1475753.0,6,14716956.20
4317,INV-341709,13808937.00,1026163.0,12,12782774.00
245,INV-112378,15022441.20,2606848.0,6,12415593.20
479,INV-124878,15382528.40,3498766.6,6,11883761.80
5615,INV-414608,13986386.80,3195330.4,8,10791056.40
3362,INV-287615,11446498.80,1118468.6,6,10328030.20
7916,INV-545524,10766690.00,529920.0,4,10236770.00
3461,INV-293862,11686159.20,1478894.4,4,10207264.80
12071,INV-777842,13715578.00,3661197.2,4,10054380.80
10394,INV-683191,10733975.06,1108205.6,6,9625769.46


In [43]:
print(
    "Total excess payment amount:",
    round(overpaid_invoices["excess_amount"].sum(), 2)
)

print(
    "Average excess per invoice:",
    round(overpaid_invoices["excess_amount"].mean(), 2)
)

print(
    "Maximum excess payment:",
    round(overpaid_invoices["excess_amount"].max(), 2)
)

Total excess payment amount: 757442104.51
Average excess per invoice: 3250824.48
Maximum excess payment: 14716956.2


In [44]:
# Merge payment summary with invoice payment status
status_check = payment_summary.merge(
    invoices[["invoice_id", "payment_status"]],
    on="invoice_id",
    how="left"
)

print("Payment status distribution:")
display(status_check["payment_status"].value_counts())

print("\nPayment status vs payment totals:")
display(
    status_check.groupby("payment_status").agg(
        total_invoices=("invoice_id", "count"),
        total_paid=("total_paid", "sum"),
        invoice_value=("invoice_amount", "sum"),
        overpaid_invoices=(
            "excess_amount",
            lambda x: (x > 0.01).sum()
        )
    )
)

Payment status distribution:


payment_status
Paid              12536
Partially Paid     3660
Unpaid               35
Name: count, dtype: int64


Payment status vs payment totals:


,total_invoices,total_paid,invoice_value,overpaid_invoices
payment_status,,,,
Paid,12536,2.150001e+10,2.023478e+10,319
Partially Paid,3660,3.281717e+09,5.969355e+09,69
Unpaid,35,9.080544e+07,4.713468e+07,28


In [45]:
unpaid_check = status_check[
    status_check["payment_status"] == "Unpaid"
].copy()

print("Unpaid invoices:", len(unpaid_check))

print(
    "Unpaid invoices with recorded payments:",
    (unpaid_check["total_paid"] > 0).sum()
)

display(
    unpaid_check[
        ["invoice_id", "total_paid", "invoice_amount",
         "excess_amount", "payment_count"]
    ].head(15)
)

Unpaid invoices: 35
Unpaid invoices with recorded payments: 35


,invoice_id,total_paid,invoice_amount,excess_amount,payment_count
745,INV-137158,487424.00,243712.0,243712.00,2
850,INV-143035,3415432.80,1707716.4,1707716.40,2
1376,INV-173464,627813.10,519773.0,108040.10,2
1382,INV-173598,77455.20,38727.6,38727.60,2
1587,INV-185110,777067.06,216172.4,560894.66,2
1771,INV-195858,89816.00,2352083.0,-2262267.00,2
2764,INV-251871,2021269.72,2029450.6,-8180.88,2
2854,INV-256846,5044716.40,2522358.2,2522358.20,2
2964,INV-262652,2290716.40,1097875.8,1192840.60,2
3250,INV-278818,214224.46,168504.0,45720.46,2


In [46]:
payments = cleaned_datasets["payments"]

print("Total payment records:", len(payments))

print(
    "Complete duplicate payment rows:",
    payments.duplicated().sum()
)

print(
    "Duplicate payment IDs:",
    payments["payment_id"].duplicated().sum()
)

print(
    "Invoices with more than one payment:",
    (payment_summary["payment_count"] > 1).sum()
)

Total payment records: 19257
Complete duplicate payment rows: 0
Duplicate payment IDs: 202
Invoices with more than one payment: 2562


In [47]:
suppliers = cleaned_datasets["suppliers"]

print("Supplier Columns:")
print(suppliers.columns.tolist())

print("\nSupplier Dataset Shape:")
print(suppliers.shape)

print("\nFirst 5 Supplier Records:")
display(suppliers.head())

Supplier Columns:
['supplier_id', 'supplier_name', 'supplier_type', 'product_category', 'city', 'province', 'region', 'pincode', 'lead_time_days', 'reliability_score', 'import_duty_rate', 'china_tax_id']

Supplier Dataset Shape:
(8, 12)

First 5 Supplier Records:


,supplier_id,supplier_name,supplier_type,product_category,city,province,region,pincode,lead_time_days,reliability_score,import_duty_rate,china_tax_id
0,SUP0001,Shenzhen OEM Supplies,OEM,Medium Parts,Shenzhen,Guangdong,South China,586808,19,5,8,90SUZHOU398241
1,SUP0002,Guangzhou Local Vendor Supplies,Local Vendor,Small Parts,Guangzhou,Guangdong,South China,966203,10,5,12,89TIANJI357980
2,SUP0003,Shanghai Distributor Supplies,Distributor,Large Parts,Shanghai,Shanghai,East China,655757,16,4,8,96SHENZH983302
3,SUP0004,Ningbo Distributor Supplies,Distributor,Large Parts,Ningbo,Zhejiang,East China,536477,18,3,8,39GUANGZ946640
4,SUP0005,Tianjin OEM Supplies,OEM,Small Parts,Tianjin,Tianjin,North China,334261,28,3,5,59GUANGZ986793


In [48]:
suppliers = cleaned_datasets["suppliers"]

print(
    "Total supplier records:",
    len(suppliers)
)

print(
    "Duplicate supplier IDs:",
    suppliers["supplier_id"].duplicated().sum()
)

print(
    "Duplicate supplier names:",
    suppliers["supplier_name"].duplicated().sum()
)

print(
    "Complete duplicate rows:",
    suppliers.duplicated().sum()
)

Total supplier records: 8
Duplicate supplier IDs: 0
Duplicate supplier names: 0
Complete duplicate rows: 0


In [49]:
print("Missing values in each column:")
display(suppliers.isnull().sum())

Missing values in each column:


supplier_id          0
supplier_name        0
supplier_type        0
product_category     0
city                 0
province             0
region               0
pincode              0
lead_time_days       0
reliability_score    0
import_duty_rate     0
china_tax_id         0
dtype: int64

In [50]:
print(
    "Negative lead times:",
    (suppliers["lead_time_days"] < 0).sum()
)

print(
    "Zero lead times:",
    (suppliers["lead_time_days"] == 0).sum()
)

print(
    "Invalid reliability scores:",
    (
        (suppliers["reliability_score"] < 1) |
        (suppliers["reliability_score"] > 5)
    ).sum()
)

print(
    "Invalid import duty rates:",
    (
        (suppliers["import_duty_rate"] < 0) |
        (suppliers["import_duty_rate"] > 100)
    ).sum()
)

Negative lead times: 0
Zero lead times: 0
Invalid reliability scores: 0
Invalid import duty rates: 0


In [51]:
print("Supplier Type Distribution:")
display(suppliers["supplier_type"].value_counts())

print("\nProduct Category Distribution:")
display(suppliers["product_category"].value_counts())

print("\nSupplier Reliability Summary:")
display(suppliers["reliability_score"].describe())

Supplier Type Distribution:


supplier_type
OEM             3
Distributor     3
Local Vendor    2
Name: count, dtype: int64


Product Category Distribution:


product_category
Small Parts     3
Large Parts     3
Medium Parts    2
Name: count, dtype: int64


Supplier Reliability Summary:


count    8.00000
mean     4.00000
std      0.92582
min      3.00000
25%      3.00000
50%      4.00000
75%      5.00000
max      5.00000
Name: reliability_score, dtype: float64

In [52]:
print("Duplicate supplier IDs:",
      suppliers["supplier_id"].duplicated().sum())

print("Duplicate supplier names:",
      suppliers["supplier_name"].duplicated().sum())

print("Complete duplicate rows:",
      suppliers.duplicated().sum())

print("\nMissing values:")
display(suppliers.isnull().sum())

Duplicate supplier IDs: 0
Duplicate supplier names: 0
Complete duplicate rows: 0

Missing values:


supplier_id          0
supplier_name        0
supplier_type        0
product_category     0
city                 0
province             0
region               0
pincode              0
lead_time_days       0
reliability_score    0
import_duty_rate     0
china_tax_id         0
dtype: int64

In [53]:
customers = cleaned_datasets["customers"]

print("Customer Columns:")
print(customers.columns.tolist())

print("\nDataset Shape:")
print(customers.shape)

print("\nFirst 5 Customer Records:")
display(customers.head())

Customer Columns:
['customer_id', 'customer_type', 'industry_segment', 'city', 'state', 'pincode', 'region', 'branch_id', 'credit_limit', 'current_balance', 'payment_terms', 'customer_since', 'last_purchase_date', 'total_purchase_value', 'customer_rating']

Dataset Shape:
(500, 15)

First 5 Customer Records:


,customer_id,customer_type,industry_segment,city,state,pincode,region,branch_id,credit_limit,current_balance,payment_terms,customer_since,last_purchase_date,total_purchase_value,customer_rating
0,C0001,Retail,Construction,Kolkata,West Bengal,991035,East,KOL001,45896,43284,Advance,2024-04-10,2024-04-25,3892818,4
1,C0002,Fleet Owner,Manufacturing,Hyderabad,Telangana,464589,South,HYD001,531506,426735,Advance,2022-01-19,2024-12-04,3168730,2
2,C0003,Government,Logistics,Surat,Gujarat,202475,West,DEL001,1306933,733495,Net 30,2017-02-14,2019-06-11,3584708,2
3,C0004,Fleet Owner,Logistics,Chennai,Tamil Nadu,576213,South,CHN001,506336,250255,Net 30,2017-12-20,2021-12-26,3810635,3
4,C0005,Dealer,Construction,Ahmedabad,Gujarat,965661,West,AHM001,380783,183927,Net 30,2023-07-27,2024-02-17,1381374,2


In [54]:
print("Total customer records:", len(customers))

print(
    "Duplicate customer IDs:",
    customers["customer_id"].duplicated().sum()
)

print(
    "Complete duplicate rows:",
    customers.duplicated().sum()
)

Total customer records: 500
Duplicate customer IDs: 0
Complete duplicate rows: 0


In [55]:
print("Missing values in each column:")
display(customers.isnull().sum())

print(
    "\nTotal missing values:",
    customers.isnull().sum().sum()
)

Missing values in each column:


customer_id             0
customer_type           0
industry_segment        0
city                    0
state                   0
pincode                 0
region                  0
branch_id               0
credit_limit            0
current_balance         0
payment_terms           0
customer_since          0
last_purchase_date      0
total_purchase_value    0
customer_rating         0
dtype: int64


Total missing values: 0


In [56]:
print("Missing values in each column:")
display(customers.isnull().sum())

print(
    "\nTotal missing values:",
    customers.isnull().sum().sum()
)

Missing values in each column:


customer_id             0
customer_type           0
industry_segment        0
city                    0
state                   0
pincode                 0
region                  0
branch_id               0
credit_limit            0
current_balance         0
payment_terms           0
customer_since          0
last_purchase_date      0
total_purchase_value    0
customer_rating         0
dtype: int64


Total missing values: 0


In [57]:
print(
    "Negative credit limits:",
    (customers["credit_limit"] < 0).sum()
)

print(
    "Negative customer balances:",
    (customers["current_balance"] < 0).sum()
)

print(
    "Negative purchase values:",
    (customers["total_purchase_value"] < 0).sum()
)

print(
    "Invalid customer ratings:",
    (
        (customers["customer_rating"] < 1) |
        (customers["customer_rating"] > 5)
    ).sum()
)

Negative credit limits: 0
Negative customer balances: 0
Negative purchase values: 0
Invalid customer ratings: 0


In [58]:
print(
    "Customers exceeding credit limit:",
    (
        customers["current_balance"] >
        customers["credit_limit"]
    ).sum()
)

Customers exceeding credit limit: 0


In [59]:
print("Customer Type Distribution:")
display(customers["customer_type"].value_counts())

print("\nIndustry Segment Distribution:")
display(customers["industry_segment"].value_counts())

print("\nRegion-wise Customer Distribution:")
display(customers["region"].value_counts())

print("\nCustomer Rating Distribution:")
display(customers["customer_rating"].value_counts().sort_index())

Customer Type Distribution:


customer_type
Government     114
Corporate      104
Dealer          99
Fleet Owner     95
Retail          88
Name: count, dtype: int64


Industry Segment Distribution:


industry_segment
Mining            98
Logistics         96
Construction      90
Infrastructure    79
Manufacturing     70
Rental            67
Name: count, dtype: int64


Region-wise Customer Distribution:


region
South         158
West          151
East           87
North          82
Central        11
North East     11
Name: count, dtype: int64


Customer Rating Distribution:


customer_rating
1    106
2     88
3    109
4    103
5     94
Name: count, dtype: int64

In [60]:
customer_financial_summary = customers[
    ["credit_limit", "current_balance", "total_purchase_value"]
].describe()

display(customer_financial_summary)

,credit_limit,current_balance,total_purchase_value
count,5.000000e+02,5.000000e+02,5.000000e+02
mean,4.915253e+05,2.466805e+05,2.480429e+06
std,3.720844e+05,2.588476e+05,1.454708e+06
min,1.055200e+04,6.300000e+01,5.173000e+03
25%,2.106442e+05,3.739025e+04,1.221794e+06
50%,4.171450e+05,1.785700e+05,2.546906e+06
75%,7.136942e+05,3.488998e+05,3.755662e+06
max,1.493816e+06,1.435105e+06,4.998438e+06


In [61]:
print("Total Credit Limit:",
      customers["credit_limit"].sum())

print("Total Current Balance:",
      customers["current_balance"].sum())

print("Total Purchase Value:",
      customers["total_purchase_value"].sum())

print("Average Customer Purchase Value:",
      customers["total_purchase_value"].mean())

Total Credit Limit: 245762629
Total Current Balance: 123340274
Total Purchase Value: 1240214657
Average Customer Purchase Value: 2480429.314


In [62]:
top_10_customers = customers.sort_values(
    by="total_purchase_value",
    ascending=False
)[
    [
        "customer_id",
        "customer_type",
        "industry_segment",
        "region",
        "credit_limit",
        "current_balance",
        "total_purchase_value",
        "customer_rating"
    ]
].head(10)

display(top_10_customers)

,customer_id,customer_type,industry_segment,region,credit_limit,current_balance,total_purchase_value,customer_rating
266,C0267,Fleet Owner,Infrastructure,South,253236,64982,4998438,3
394,C0395,Corporate,Manufacturing,East,680072,209294,4995587,1
416,C0417,Fleet Owner,Mining,East,587793,383902,4991661,3
433,C0434,Corporate,Mining,Central,437713,377663,4961157,4
274,C0275,Fleet Owner,Construction,West,651275,262408,4940277,5
320,C0321,Retail,Logistics,South,33291,18767,4937062,2
411,C0412,Corporate,Infrastructure,East,699539,571001,4934990,3
207,C0208,Fleet Owner,Logistics,North,737567,537907,4909280,4
451,C0452,Fleet Owner,Mining,South,362507,335242,4909073,1
118,C0119,Government,Manufacturing,West,1079453,124485,4897663,4


In [63]:
customers["credit_utilization"] = (
    customers["current_balance"] /
    customers["credit_limit"]
) * 100

high_credit_utilization = customers[
    customers["credit_utilization"] >= 80
].sort_values(
    by="credit_utilization",
    ascending=False
)

print("Customers with 80% or more credit utilization:",
      len(high_credit_utilization))

display(
    high_credit_utilization[
        [
            "customer_id",
            "customer_type",
            "credit_limit",
            "current_balance",
            "credit_utilization"
        ]
    ].head(15)
)

Customers with 80% or more credit utilization: 112


,customer_id,customer_type,credit_limit,current_balance,credit_utilization
200,C0201,Dealer,468119,467851,99.942750
342,C0343,Corporate,994769,993939,99.916564
227,C0228,Government,1026411,1025544,99.915531
71,C0072,Government,1196826,1192089,99.604203
369,C0370,Corporate,375748,373898,99.507649
7,C0008,Corporate,941092,934117,99.258840
263,C0264,Retail,35955,35640,99.123905
423,C0424,Retail,28345,28072,99.036867
163,C0164,Government,1454565,1435105,98.662143
17,C0018,Retail,49835,49052,98.428815


In [64]:
import numpy as np

customers["credit_category"] = np.select(
    [
        customers["credit_utilization"] < 50,
        customers["credit_utilization"] < 80
    ],
    [
        "Low",
        "Medium"
    ],
    default="High"
)

print("Customer Credit Risk Categories:")

display(
    customers["credit_category"].value_counts()
)

print("\nPercentage Distribution:")

display(
    customers["credit_category"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Customer Credit Risk Categories:


credit_category
Low       259
Medium    129
High      112
Name: count, dtype: int64


Percentage Distribution:


credit_category
Low       51.8
Medium    25.8
High      22.4
Name: proportion, dtype: float64

In [65]:
region_analysis = customers.groupby("region").agg(
    total_customers=("customer_id", "count"),
    total_purchase_value=("total_purchase_value", "sum"),
    average_purchase_value=("total_purchase_value", "mean"),
    total_current_balance=("current_balance", "sum"),
    average_customer_rating=("customer_rating", "mean")
).round(2)

display(
    region_analysis.sort_values(
        by="total_purchase_value",
        ascending=False
    )
)

,total_customers,total_purchase_value,average_purchase_value,total_current_balance,average_customer_rating
region,,,,,
South,158,397691892,2517037.29,38981864,2.99
West,151,369039405,2443969.57,40106409,2.96
East,87,221847108,2549966.76,21725281,3.02
North,82,191513829,2335534.50,17881033,3.09
Central,11,35346221,3213292.82,2428086,2.45
North East,11,24776202,2252382.00,2217601,2.55


In [66]:
industry_analysis = customers.groupby("industry_segment").agg(
    total_customers=("customer_id", "count"),
    total_purchase_value=("total_purchase_value", "sum"),
    average_purchase_value=("total_purchase_value", "mean"),
    total_current_balance=("current_balance", "sum"),
    average_customer_rating=("customer_rating", "mean")
).round(2)

display(
    industry_analysis.sort_values(
        by="total_purchase_value",
        ascending=False
    )
)

,total_customers,total_purchase_value,average_purchase_value,total_current_balance,average_customer_rating
industry_segment,,,,,
Construction,90,252822390,2809137.67,22938116,3.08
Mining,98,245832919,2508499.17,26636491,2.95
Logistics,96,236453753,2463059.93,24840817,2.90
Infrastructure,79,179395682,2270831.42,20169170,3.27
Rental,67,169935465,2536350.22,14947563,3.04
Manufacturing,70,155774448,2225349.26,13808117,2.64


In [67]:
customer_type_analysis = customers.groupby("customer_type").agg(
    total_customers=("customer_id", "count"),
    total_purchase_value=("total_purchase_value", "sum"),
    average_purchase_value=("total_purchase_value", "mean"),
    total_current_balance=("current_balance", "sum"),
    average_customer_rating=("customer_rating", "mean")
).round(2)

display(
    customer_type_analysis.sort_values(
        by="total_purchase_value",
        ascending=False
    )
)

,total_customers,total_purchase_value,average_purchase_value,total_current_balance,average_customer_rating
customer_type,,,,,
Government,114,277088353,2430599.59,55258724,3.11
Corporate,104,264949950,2547595.67,30151279,2.97
Fleet Owner,95,239873468,2524983.87,22311400,2.99
Dealer,99,239794969,2422171.40,14365701,2.98
Retail,88,218507917,2483044.51,1253170,2.83


In [68]:
branches = cleaned_datasets["branches"]

print("Total branches:", len(branches))
print("Total columns:", len(branches.columns))

print("\nColumn names:")
print(branches.columns.tolist())

print("\nFirst 5 records:")
display(branches.head())

print("\nDataset information:")
branches.info()

Total branches: 6
Total columns: 13

Column names:
['branch_id', 'branch_name', 'city', 'state', 'region', 'warehouse_type', 'warehouse_capacity', 'service_center_available', 'manager_id', 'total_employees', 'avg_monthly_revenue', 'monthly_operational_cost', 'market_demand_index']

First 5 records:


,branch_id,branch_name,city,state,region,warehouse_type,warehouse_capacity,service_center_available,manager_id,total_employees,avg_monthly_revenue,monthly_operational_cost,market_demand_index
0,DEL001,Delhi Central,Delhi,Delhi,North,Central,45230.0,Yes,9593,92,5234890,3189000,9
1,PUN001,Pune Distribution,Pune,Maharashtra,West,Regional,38750.0,Yes,6114,74,4389120,2654000,8
2,CHN001,Chennai South Hub,Chennai,Tamil Nadu,South,Regional,41890.0,Yes,7922,81,4567780,2739000,8
3,HYD001,Hyderabad Logistics,Hyderabad,Telangana,South,Branch,36210.0,No,4390,57,3102455,1887000,7
4,KOL001,Kolkata East Depot,Kolkata,West Bengal,East,Branch,29880.0,No,5981,49,2894770,1743000,6



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   branch_id                 6 non-null      object 
 1   branch_name               6 non-null      object 
 2   city                      6 non-null      object 
 3   state                     6 non-null      object 
 4   region                    6 non-null      object 
 5   warehouse_type            6 non-null      object 
 6   warehouse_capacity        6 non-null      float64
 7   service_center_available  6 non-null      object 
 8   manager_id                6 non-null      int64  
 9   total_employees           6 non-null      int64  
 10  avg_monthly_revenue       6 non-null      int64  
 11  monthly_operational_cost  6 non-null      int64  
 12  market_demand_index       6 non-null      int64  
dtypes: float64(1), int64(5), object(7)
memory usage

In [69]:
branches = cleaned_datasets["branches"]

print("Duplicate branch IDs:",
      branches["branch_id"].duplicated().sum())

print("Duplicate branch names:",
      branches["branch_name"].duplicated().sum())

print("Complete duplicate rows:",
      branches.duplicated().sum())

print("\nMissing values in each column:")
print(branches.isnull().sum())

print("\nTotal missing values:",
      branches.isnull().sum().sum())

Duplicate branch IDs: 0
Duplicate branch names: 0
Complete duplicate rows: 0

Missing values in each column:
branch_id                   0
branch_name                 0
city                        0
state                       0
region                      0
warehouse_type              0
warehouse_capacity          0
service_center_available    0
manager_id                  0
total_employees             0
avg_monthly_revenue         0
monthly_operational_cost    0
market_demand_index         0
dtype: int64

Total missing values: 0


In [70]:
print("Negative warehouse capacity:",
      (branches["warehouse_capacity"] < 0).sum())

print("Zero warehouse capacity:",
      (branches["warehouse_capacity"] == 0).sum())

print("Negative employees:",
      (branches["total_employees"] < 0).sum())

print("Zero employees:",
      (branches["total_employees"] == 0).sum())

print("Negative monthly revenue:",
      (branches["avg_monthly_revenue"] < 0).sum())

print("Negative operational cost:",
      (branches["monthly_operational_cost"] < 0).sum())

print("Invalid market demand index:",
      ((branches["market_demand_index"] < 1) |
       (branches["market_demand_index"] > 10)).sum())

print("Invalid service center values:",
      (~branches["service_center_available"].isin(["Yes", "No"])).sum())

Negative warehouse capacity: 0
Zero warehouse capacity: 0
Negative employees: 0
Zero employees: 0
Negative monthly revenue: 0
Negative operational cost: 0
Invalid market demand index: 0
Invalid service center values: 0


In [71]:
print("1. Branches by Region")
display(branches["region"].value_counts())

print("\n2. Branches by Warehouse Type")
display(branches["warehouse_type"].value_counts())

print("\n3. Service Centre Availability")
display(branches["service_center_available"].value_counts())

print("\n4. Branches by State")
display(branches["state"].value_counts())

1. Branches by Region


region
West     2
South    2
North    1
East     1
Name: count, dtype: int64


2. Branches by Warehouse Type


warehouse_type
Regional       2
Branch         2
Central        1
Service Hub    1
Name: count, dtype: int64


3. Service Centre Availability


service_center_available
Yes    4
No     2
Name: count, dtype: int64


4. Branches by State


state
Delhi          1
Maharashtra    1
Tamil Nadu     1
Telangana      1
West Bengal    1
Gujarat        1
Name: count, dtype: int64

In [72]:
branch_finance = branches.copy()

branch_finance["operating_profit"] = (
    branch_finance["avg_monthly_revenue"] -
    branch_finance["monthly_operational_cost"]
)

branch_finance["profit_margin_percentage"] = (
    branch_finance["operating_profit"] /
    branch_finance["avg_monthly_revenue"] * 100
)

display(
    branch_finance[
        [
            "branch_name",
            "avg_monthly_revenue",
            "monthly_operational_cost",
            "operating_profit",
            "profit_margin_percentage"
        ]
    ].sort_values("operating_profit", ascending=False)
)

,branch_name,avg_monthly_revenue,monthly_operational_cost,operating_profit,profit_margin_percentage
0,Delhi Central,5234890,3189000,2045890,39.081815
2,Chennai South Hub,4567780,2739000,1828780,40.036517
1,Pune Distribution,4389120,2654000,1735120,39.532298
5,Ahmedabad West Hub,3245890,2018000,1227890,37.829070
3,Hyderabad Logistics,3102455,1887000,1215455,39.177200
4,Kolkata East Depot,2894770,1743000,1151770,39.787962


In [73]:
print("Total warehouse capacity:",
      branches["warehouse_capacity"].sum())

print("Average warehouse capacity:",
      branches["warehouse_capacity"].mean())

print("Highest warehouse capacity:",
      branches["warehouse_capacity"].max())

print("Lowest warehouse capacity:",
      branches["warehouse_capacity"].min())

print("\nWarehouse capacity by branch:")

display(
    branches[
        ["branch_name", "warehouse_type", "warehouse_capacity"]
    ].sort_values("warehouse_capacity", ascending=False)
)

Total warehouse capacity: 226080.0
Average warehouse capacity: 37680.0
Highest warehouse capacity: 45230.0
Lowest warehouse capacity: 29880.0

Warehouse capacity by branch:


,branch_name,warehouse_type,warehouse_capacity
0,Delhi Central,Central,45230.0
2,Chennai South Hub,Regional,41890.0
1,Pune Distribution,Regional,38750.0
3,Hyderabad Logistics,Branch,36210.0
5,Ahmedabad West Hub,Service Hub,34120.0
4,Kolkata East Depot,Branch,29880.0


In [74]:
branch_summary = branches.copy()

branch_summary["operating_profit"] = (
    branch_summary["avg_monthly_revenue"] -
    branch_summary["monthly_operational_cost"]
)

branch_summary["profit_margin"] = (
    branch_summary["operating_profit"] /
    branch_summary["avg_monthly_revenue"] * 100
)

display(
    branch_summary[
        [
            "branch_name",
            "region",
            "total_employees",
            "warehouse_capacity",
            "avg_monthly_revenue",
            "monthly_operational_cost",
            "operating_profit",
            "profit_margin"
        ]
    ].sort_values("avg_monthly_revenue", ascending=False)
)

,branch_name,region,total_employees,warehouse_capacity,avg_monthly_revenue,monthly_operational_cost,operating_profit,profit_margin
0,Delhi Central,North,92,45230.0,5234890,3189000,2045890,39.081815
2,Chennai South Hub,South,81,41890.0,4567780,2739000,1828780,40.036517
1,Pune Distribution,West,74,38750.0,4389120,2654000,1735120,39.532298
5,Ahmedabad West Hub,West,63,34120.0,3245890,2018000,1227890,37.829070
3,Hyderabad Logistics,South,57,36210.0,3102455,1887000,1215455,39.177200
4,Kolkata East Depot,East,49,29880.0,2894770,1743000,1151770,39.787962


In [75]:
branch_productivity = branches.copy()

branch_productivity["operating_profit"] = (
    branch_productivity["avg_monthly_revenue"] -
    branch_productivity["monthly_operational_cost"]
)

branch_productivity["revenue_per_employee"] = (
    branch_productivity["avg_monthly_revenue"] /
    branch_productivity["total_employees"]
)

branch_productivity["profit_per_employee"] = (
    branch_productivity["operating_profit"] /
    branch_productivity["total_employees"]
)

display(
    branch_productivity[
        [
            "branch_name",
            "total_employees",
            "avg_monthly_revenue",
            "operating_profit",
            "revenue_per_employee",
            "profit_per_employee"
        ]
    ].sort_values("revenue_per_employee", ascending=False)
)

,branch_name,total_employees,avg_monthly_revenue,operating_profit,revenue_per_employee,profit_per_employee
1,Pune Distribution,74,4389120,1735120,59312.432432,23447.567568
4,Kolkata East Depot,49,2894770,1151770,59076.938776,23505.510204
0,Delhi Central,92,5234890,2045890,56900.978261,22237.934783
2,Chennai South Hub,81,4567780,1828780,56392.345679,22577.530864
3,Hyderabad Logistics,57,3102455,1215455,54429.035088,21323.771930
5,Ahmedabad West Hub,63,3245890,1227890,51522.063492,19490.317460


In [76]:
inventory = cleaned_datasets["inventory_master"]

print("Total inventory records:", len(inventory))
print("Total columns:", len(inventory.columns))

print("\nColumn names:")
print(inventory.columns.tolist())

print("\nFirst 10 records:")
display(inventory.head(10))

print("\nDataset information:")
inventory.info()

Total inventory records: 180
Total columns: 8

Column names:
['product_id', 'branch_id', 'opening_stock', 'reorder_level', 'safety_stock', 'max_stock', 'current_stock', 'warehouse_bin']

First 10 records:


,product_id,branch_id,opening_stock,reorder_level,safety_stock,max_stock,current_stock,warehouse_bin
0,P001,DEL001,196,59,46,340,115928,C15
1,P001,PUN001,198,70,43,349,109389,D16
2,P001,CHN001,187,62,44,311,99373,A11
3,P001,HYD001,226,71,65,417,108772,A10
4,P001,KOL001,263,94,58,475,103808,F03
5,P001,AHM001,236,77,53,455,106714,C20
6,P002,DEL001,166,57,35,261,108628,A06
7,P002,PUN001,113,41,27,180,101623,E01
8,P002,CHN001,267,102,76,465,105885,B04
9,P002,HYD001,219,78,64,383,121013,A02



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_id     180 non-null    object
 1   branch_id      180 non-null    object
 2   opening_stock  180 non-null    int64 
 3   reorder_level  180 non-null    int64 
 4   safety_stock   180 non-null    int64 
 5   max_stock      180 non-null    int64 
 6   current_stock  180 non-null    int64 
 7   warehouse_bin  180 non-null    object
dtypes: int64(5), object(3)
memory usage: 11.4+ KB


In [77]:
branch_inventory = inventory.groupby("branch_id").agg(
    total_products=("product_id", "nunique"),
    opening_stock=("opening_stock", "sum"),
    current_stock=("current_stock", "sum"),
    reorder_level=("reorder_level", "sum"),
    safety_stock=("safety_stock", "sum"),
    max_stock=("max_stock", "sum")
).reset_index()

display(branch_inventory)

,branch_id,total_products,opening_stock,current_stock,reorder_level,safety_stock,max_stock
0,AHM001,30,5444,3154992,1915,1319,9690
1,CHN001,30,5472,3234245,1952,1353,9642
2,DEL001,30,5506,3288670,1968,1379,9744
3,HYD001,30,5553,3245291,1928,1428,9437
4,KOL001,30,6063,3123805,2174,1498,10846
5,PUN001,30,4931,3256263,1682,1160,8557


In [78]:
inventory_summary = inventory[
    ["opening_stock", "current_stock",
     "reorder_level", "safety_stock", "max_stock"]
].agg(["sum", "mean", "min", "max"])

display(inventory_summary)

,opening_stock,current_stock,reorder_level,safety_stock,max_stock
sum,32969.000000,1.930327e+07,11619.00,8137.000000,57916.000000
mean,183.161111,1.072404e+05,64.55,45.205556,321.755556
min,80.000000,8.885300e+04,24.00,16.000000,131.000000
max,300.000000,1.210130e+05,119.00,85.000000,582.000000


In [79]:
product_inventory = inventory.groupby("product_id").agg(
    total_branches=("branch_id", "nunique"),
    total_opening_stock=("opening_stock", "sum"),
    total_current_stock=("current_stock", "sum"),
    total_reorder_level=("reorder_level", "sum"),
    total_safety_stock=("safety_stock", "sum"),
    total_max_stock=("max_stock", "sum")
).reset_index()

product_inventory = product_inventory.merge(
    products[["product_id", "product_name", "category"]],
    on="product_id",
    how="left"
)

display(
    product_inventory.sort_values(
        "total_current_stock", ascending=False
    )
)

,product_id,total_branches,total_opening_stock,total_current_stock,total_reorder_level,total_safety_stock,total_max_stock,product_name,category
11,P012,6,1008,663107,361,254,1836,Brake Pad BP-40,Brakes
13,P014,6,1023,661663,374,241,1918,Transmission Assembly TA-800,Transmission
22,P023,6,911,661520,336,211,1583,Idler Wheel IW-55,Undercarriage
9,P010,6,890,661248,301,230,1524,Turbocharger TB-900,Engine
16,P017,6,1199,659066,412,334,2168,Fan Belt FB-30,Belts
4,P005,6,1175,658820,433,302,2108,Fuel Injector FI-220,Engine
6,P007,6,1383,656823,464,324,2529,Hydraulic Hose HH-60,Hydraulic
23,P024,6,1223,656188,432,276,1987,Track Roller TR-60,Undercarriage
1,P002,6,1155,655012,419,302,1979,Engine Oil Filter OF-90,Filters
24,P025,6,1209,651781,437,328,2134,Cabin Glass CG-20,Cabin


In [80]:
category_inventory = inventory.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left"
)

category_summary = category_inventory.groupby("category").agg(
    total_products=("product_id", "nunique"),
    total_current_stock=("current_stock", "sum"),
    total_opening_stock=("opening_stock", "sum"),
    average_current_stock=("current_stock", "mean")
).reset_index()

display(category_summary)

,category,total_products,total_current_stock,total_opening_stock,average_current_stock
0,Attachments,1,624694,1313,104115.666667
1,Belts,1,659066,1199,109844.333333
2,Brakes,1,663107,1008,110517.833333
3,Cabin,1,651781,1209,108630.166667
4,Cooling,1,633960,985,105660.000000
5,Electrical,3,1903341,3339,105741.166667
6,Engine,4,2584941,4138,107705.875000
7,Filters,2,1305127,2421,108760.583333
8,Hydraulic,6,3851410,7289,106983.611111
9,Lubricants,1,631316,1027,105219.333333


In [81]:
inventory_analysis = inventory.copy()

inventory_analysis["stock_change"] = (
    inventory_analysis["current_stock"] -
    inventory_analysis["opening_stock"]
)

print("Total opening stock:",
      inventory_analysis["opening_stock"].sum())

print("Total current stock:",
      inventory_analysis["current_stock"].sum())

print("Total stock change:",
      inventory_analysis["stock_change"].sum())

print("Records with increased stock:",
      (inventory_analysis["stock_change"] > 0).sum())

print("Records with decreased stock:",
      (inventory_analysis["stock_change"] < 0).sum())

print("Records with unchanged stock:",
      (inventory_analysis["stock_change"] == 0).sum())

Total opening stock: 32969
Total current stock: 19303266
Total stock change: 19270297
Records with increased stock: 180
Records with decreased stock: 0
Records with unchanged stock: 0


In [82]:
inventory_analysis["reorder_status"] = (
    inventory_analysis.apply(
        lambda row: "Reorder Required"
        if row["current_stock"] <= row["reorder_level"]
        else "Stock Available",
        axis=1
    )
)

display(
    inventory_analysis["reorder_status"].value_counts()
)

reorder_status
Stock Available    180
Name: count, dtype: int64

In [83]:
inventory_analysis["safety_stock_status"] = (
    inventory_analysis.apply(
        lambda row: "Below Safety Stock"
        if row["current_stock"] < row["safety_stock"]
        else "Above Safety Stock",
        axis=1
    )
)

display(
    inventory_analysis["safety_stock_status"].value_counts()
)

safety_stock_status
Above Safety Stock    180
Name: count, dtype: int64

In [84]:
inventory_analysis["stock_limit_status"] = (
    inventory_analysis.apply(
        lambda row: "Above Maximum"
        if row["current_stock"] > row["max_stock"]
        else "Within Maximum",
        axis=1
    )
)

display(
    inventory_analysis["stock_limit_status"].value_counts()
)

print("\nTotal excess stock units:",
      (inventory_analysis["current_stock"] -
       inventory_analysis["max_stock"]).clip(lower=0).sum())

stock_limit_status
Above Maximum    180
Name: count, dtype: int64


Total excess stock units: 19245350


In [85]:
print("Total warehouse bins:",
      inventory["warehouse_bin"].nunique())

print("\nInventory records by warehouse bin:")
display(inventory["warehouse_bin"].value_counts())

print("\nWarehouse bin stock summary:")
display(
    inventory.groupby("warehouse_bin").agg(
        total_products=("product_id", "nunique"),
        total_current_stock=("current_stock", "sum")
    ).reset_index()
)

Total warehouse bins: 95

Inventory records by warehouse bin:


warehouse_bin
E17    5
F08    5
B09    4
A15    4
B15    3
      ..
C06    1
D02    1
C19    1
F05    1
A05    1
Name: count, Length: 95, dtype: int64


Warehouse bin stock summary:


,warehouse_bin,total_products,total_current_stock
0,A01,3,313089
1,A02,3,334263
2,A03,2,214282
3,A05,1,112388
4,A06,2,210040
...,...,...,...
90,F16,1,102706
91,F17,1,109625
92,F18,1,107584
93,F19,3,328008


In [87]:
inventory = cleaned_datasets["inventory_master"]

In [88]:
branch_stock_analysis = inventory.copy()

branch_stock_analysis["excess_stock"] = (
    branch_stock_analysis["current_stock"] -
    branch_stock_analysis["max_stock"]
).clip(lower=0)

branch_stock_summary = branch_stock_analysis.groupby(
    "branch_id"
).agg(
    total_products=("product_id", "nunique"),
    total_current_stock=("current_stock", "sum"),
    total_max_stock=("max_stock", "sum"),
    total_excess_stock=("excess_stock", "sum")
).reset_index()

display(
    branch_stock_summary.sort_values(
        "total_excess_stock", ascending=False
    )
)

,branch_id,total_products,total_current_stock,total_max_stock,total_excess_stock
2,DEL001,30,3288670,9744,3278926
5,PUN001,30,3256263,8557,3247706
3,HYD001,30,3245291,9437,3235854
1,CHN001,30,3234245,9642,3224603
0,AHM001,30,3154992,9690,3145302
4,KOL001,30,3123805,10846,3112959
